In [1]:
# A Conditional Workflow in LangGraph means the next node is chosen based on a condition.
# Think of it like an if-else statement inside your graph.

In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_community.chat_models import ChatOllama

c:\Users\Swati\llm_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model = ChatOllama(model="llama3", temparature=1.5)

C:\Users\Swati\AppData\Local\Temp\ipykernel_1052\3307222411.py:1: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  model = ChatOllama(model="llama3", temparature=1.5)


In [ ]:
class GraphState(TypedDict):
    feedback: str
    sentiment: str

In [ ]:
def get_sentiment(state: GraphState) -> str:
    feedback = state['feedback']
    prompt = f"Analyze the sentiment only as a Positive or Negative: {feedback}"
    response = model.invoke(prompt).content
    state['sentiment']  = response

# Positive path
def thank_you(state):
    return {
        "result": "Thank you for your positive feedback!"
    }

# Negative path
def escalate(state):
    return {
        "result": "Creating support ticket..."
    }

# Routing function
def route_sentiment(state):
    if "positive" in state["sentiment"].lower():
        return "positive_node"
    return "negative_node"

In [ ]:
# Create graph
graph = StateGraph(GraphState)

# Create Node
graph.add_node('get_sentiment', get_sentiment)

# Define edges
graph.add_edge(START, 'get_sentiment')
graph.add_edge('get_sentiment', END)

# compile graph
workflow = graph.compile()

In [ ]:
# Create graph
graph = StateGraph(GraphState)

graph.add_node('get_sentiment', get_sentiment)
graph.add_node('positive_node', thank_you)
graph.add_node('negative_node', escalate)
graph.add_edge(START, 'get_sentiment')
graph.add_conditional_edges(
    'get_sentiment',
    route_sentiment,
    {
        "positive_node": "positive_node",
        "negative_node": "negative_node"
    }

)
graph.add_edge('positive_node', END)
graph.add_edge('negative_node', END)    

# compile graph
workflow = graph.compile()
# Run graph
initial_state = { "feedback": "Your service is excellent." }
final_state = workflow(initial_state)
print(final_state)
